# 🗼 Watchtower Server — Google Colab Deploy

Starts the **Watchtower headless server** and exposes it publicly via **ngrok**.

1. Fill in `API_KEY` and `NGROK_TOKEN` below
2. `Runtime → Run all`
3. Copy the URL printed at the end
4. Paste into Watchtower app → **Settings → Remote Server**

> ⚠️ Session expires when the tab closes. For persistent hosting use Railway or Render.

In [ ]:
# @title ⚙️ Configuration { display-mode: "form" }
API_KEY    = "changeme"  # @param {type:"string"}
NGROK_TOKEN = ""          # @param {type:"string"} https://dashboard.ngrok.com/
PORT        = 8080        # @param {type:"integer"}

In [ ]:
# @title 1️⃣ Install Node.js 20
import subprocess
def run(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if r.returncode != 0: raise RuntimeError(r.stderr)
    return r.stdout
run("curl -fsSL https://deb.nodesource.com/setup_20.x | bash -")
run("apt-get install -y nodejs")
print("Node:", run("node --version").strip(), "| npm:", run("npm --version").strip())

In [ ]:
# @title 2️⃣ Clone repo & install dependencies
import os
SERVER_DIR = "/content/watchtower/server"
if not os.path.exists(SERVER_DIR):
    run("git clone --depth=1 https://github.com/ferelking242/watchtower.git /content/watchtower")
run("npm ci --omit=dev", cwd=SERVER_DIR)
print("Done!")

In [ ]:
# @title 3️⃣ Start the server
import subprocess, os, time, urllib.request
SERVER_DIR = "/content/watchtower/server"
os.makedirs("/content/wt-cache", exist_ok=True)
os.makedirs("/content/wt-prefs", exist_ok=True)
env = {**os.environ, "API_KEY": API_KEY, "PORT": str(PORT),
       "CACHE_DIR": "/content/wt-cache", "PREFS_DIR": "/content/wt-prefs"}
proc = subprocess.Popen(["node", "server.js"], cwd=SERVER_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)
if proc.poll() is not None: raise RuntimeError(proc.stdout.read().decode())
print("Health:", urllib.request.urlopen(f"http://localhost:{PORT}/api/ping").read().decode())
print(f"✅ Server up on port {PORT}")

In [ ]:
# @title 4️⃣ Expose via ngrok
run("pip install pyngrok -q")
from pyngrok import ngrok, conf
if NGROK_TOKEN: conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(PORT, "http")
print("\n" + "="*60)
print(f"🗼 Server URL : {tunnel.public_url}")
print(f"🔑 API Key   : {API_KEY}")
print("="*60)
print("\nApp → Settings → Remote Server → paste URL above")

In [ ]:
# @title 5️⃣ Keep-alive (optional)
import time
print("Running — press ■ to stop.")
try:
    while True: time.sleep(60); print(".", end="", flush=True)
except KeyboardInterrupt:
    proc.terminate(); ngrok.kill(); print("\nStopped.")